# 05 — Exploration du retrieval : question -> top-k chunks

**Objectif** : juger à l'œil, dès maintenant, si la recherche dense est cohérente ou franchement mauvaise. Bien avant l'évaluation formelle.

1. connexion et encodeur (le même que pour l'indexation) ;
2. **garde-fou** : la base a-t-elle été indexée avec ce modèle ?
3. la requête top-k et son score ;
4. tes questions, avec le **texte complet** des chunks retrouvés ;
5. vue d'ensemble : scores du top-1 et du top-5, écart entre les deux premiers ;
6. distribution des scores sur tout le corpus : le top-1 se détache-t-il du lot ?
7. recherche exacte (numpy) contre recherche HNSW (pgvector) ;
8. premier aperçu chiffré (optionnel) : hit@k sur tes propres attentes.

Prérequis : `docker compose up -d` et une base indexée (`uv run indexer-regles`).

## 1. Connexion et encodeur

La table existe déjà : un simple `register_vector` suffit (pas besoin de réappliquer le schéma). L'encodeur est chargé par le code propre de l'incrément 2, avec la même config que l'indexation.

In [1]:
import time

import numpy as np
import psycopg
from dotenv import load_dotenv
from pgvector.psycopg import register_vector
from psycopg.rows import dict_row

from assistant_regles.ingest.config import trouver_racine
from assistant_regles.rag.config import charger_config
from assistant_regles.rag.embeddings import EncodeurBGEM3
from assistant_regles.rag.store import TABLE

load_dotenv(trouver_racine() / ".env")
conn = psycopg.connect(autocommit=True)
register_vector(conn)
print(conn.execute(f"SELECT count(*) FROM {TABLE}").fetchone()[0], "chunks en base")

207 chunks en base


In [2]:
encodeur = EncodeurBGEM3.charger(charger_config().embeddings)
print(encodeur.identifiant)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BAAI/bge-m3@5617a9f61b028005a4858fdac845db406aefb181


## 2. Garde-fou : même modèle en base et pour la question

Un vecteur de question n'est comparable qu'à des vecteurs du **même** modèle. Changer de modèle ou de révision sans réindexer produirait des résultats absurdes, sans aucune erreur. La colonne `modele_embedding` permet de le vérifier.

In [18]:
modeles = [r[0] for r in conn.execute(f"SELECT DISTINCT modele_embedding FROM {TABLE}")]
print("modèle(s) en base :", modeles)
assert modeles == [encodeur.identifiant], "Base indexée avec un autre modèle : relancer indexer-regles"

modèle(s) en base : ['BAAI/bge-m3@5617a9f61b028005a4858fdac845db406aefb181']


## 3. La requête top-k

- `embedding <=> q` est la **distance** cosinus (0 = même direction). On trie dessus, par ordre croissant.
- Le **score** affiché est `1 - distance`, c'est-à-dire la **similarité** cosinus : plus il est haut, plus le chunk est proche.
- Le paramètre nommé `%(q)s` apparaît deux fois : psycopg envoie la même valeur aux deux endroits.

Les deux durées mesurées séparent le coût de l'encodage de la question et celui de la requête SQL.

In [19]:
SQL_RECHERCHE = f"""SELECT id, code, section_num, section_titre, sous_section,
       page_debut, page_fin, type_contenu, texte,
       1 - (embedding <=> %(q)s) AS score
FROM {TABLE}
ORDER BY embedding <=> %(q)s
LIMIT %(k)s"""


def rechercher(question, k=5):
    """Renvoie (résultats, vecteur de la question, (durée encodage, durée SQL))."""
    debut = time.perf_counter()
    vecteur = encodeur.encoder([question])[0]
    duree_encodage = time.perf_counter() - debut

    debut = time.perf_counter()
    with conn.cursor(row_factory=dict_row) as cur:
        resultats = cur.execute(SQL_RECHERCHE, {"q": vecteur, "k": k}).fetchall()
    return resultats, vecteur, (duree_encodage, time.perf_counter() - debut)


def afficher(question, resultats, durees=None, texte_complet=True):
    """Affiche les résultats : rang, score, code, section, pages, puis le texte."""
    print("=" * 100)
    print("QUESTION :", question)
    if durees:
        print(f"(encodage {durees[0] * 1000:.0f} ms, SQL {durees[1] * 1000:.1f} ms)")
    print("=" * 100)
    for rang, r in enumerate(resultats, 1):
        pages = f"p. {r['page_debut']}" + (f"-{r['page_fin']}" if r["page_fin"] != r["page_debut"] else "")
        section = f"{r['section_num'] or '--'} {r['section_titre'] or ''}"
        print(f"\n#{rang}  score {r['score']:.3f}  |  {r['code'] or '(sans code)'}  |  "
              f"{section} > {r['sous_section'] or ''}  |  {pages}  |  {r['type_contenu']}")
        print("-" * 100)
        print(r["texte"] if texte_complet else r["texte"][:200] + " …")
    print()

## 4. Questions pour évaluation empirique

Les questions : 

- des questions **simples**, dont la réponse tient dans une seule sous-section ;
- des questions **formulées autrement** que le livre (synonymes, langage de joueur) ;
- des **règles proches**, qu'on pourrait confondre ;
- des questions **multi-sections** ;
- au moins une question **hors corpus**. La recherche renvoie **toujours** k résultats, même quand rien n'est pertinent. Ses scores servent de point de comparaison.

In [42]:
QUESTIONS_BASIQUES = [
    "Quelle distance une unité peut-elle parcourir pendant sa phase de mouvement ?",
    "Comment se déroule une charge ?",
    "Une unité engagée au corps à corps peut-elle tirer ?",
    "Comment utilise-t-on un stratagème ?",
    "Que se passe-t-il quand une figurine perd son dernier point de vie ?",
    "Quelle différence entre avancer et battre en retraite ?",   # règles proches
    "Quelle est la recette de la pâte à crêpes ?",                # hors corpus
]
QUESTIONS_COMPLEXES = [
    "Que doit faire une unitée en fuite ?",
    "Comment alouer les dégats sur une unité composée de modèles avec différentes endurances et sauvegardes?",
    "Quel est le commandement d'une unité à plusieurs figurines ayant des statistiques de commandemant différentes?",
    "Les pistolets peuvent-ils être utilisés au corps à corps?",
    "Qui est le président de la planète terre?"
]

In [43]:
QUESTIONS = QUESTIONS_COMPLEXES

In [44]:
# Une question à la fois, texte complet (change l'indice pour passer à la suivante)
question = QUESTIONS[4]
resultats, _, durees = rechercher(question, k=5)
afficher(question, resultats, durees)

QUESTION : Qui est le président de la planète terre?
(encodage 9 ms, SQL 1.3 ms)

#1  score 0.318  |  14.02  |  14 Objectif > NIVEAU DE CONTRÔLE  |  p. 52  |  regle
----------------------------------------------------------------------------------------------------
CHAMPS DE BATAILLE ET TACTIQUES > 14 Objectif > 14.02 NIVEAU DE CONTRÔLE

Au début de la bataille, aucun joueur ne contrôle d'objectif du champ de bataille. Pour prendre le contrôle d'un objectif, un joueur aura besoin d'une ou plusieurs figurines ayant une caractéristique de CO de 1 ou plus à portée de lui. Une figurine est à portée d'un objectif de terrain tant qu'elle est dans cette zone de terrain.
À la fin de chaque phase et de chaque tour, pour déterminer le niveau de contrôle d'un joueur sur un objectif, additionnez les caractéristiques de CO de toutes les figurines de l'armée de ce joueur qui sont à portée de cet objectif :
Le joueur qui a le niveau de contrôle le plus élevé sur cet objectif contrôle l'objectif.
Si l

In [45]:
# Toutes les questions d'un coup (texte_complet=False pour un aperçu plus court)
for question in QUESTIONS:
    resultats, _, durees = rechercher(question, k=5)
    afficher(question, resultats, durees, texte_complet=True)

QUESTION : Que doit faire une unitée en fuite ?
(encodage 6 ms, SQL 1.7 ms)

#1  score 0.549  |  09.07  |  09 Phase de mouvement > MOUVEMENT DE RETRAITE  |  p. 33  |  regle
----------------------------------------------------------------------------------------------------
LE ROUND DE BATAILLE > 09 Phase de mouvement > 09.07 MOUVEMENT DE RETRAITE

DISTANCE MAXIMALE : La caractéristique de M de votre unité.
ÉLIGIBLE SI : Votre unité est Engagée.
EFFET : Votre unité se déplace comme décrit dans Mouvement (03).
AVANT DE SE DÉPLACER : Choisissez le mode de retraite :
-› Retraite en Bon Ordre : Si votre unité n'est pas ébranlée, vous pouvez choisir ce mode.
-› Fuite Désespérée : Sinon, vous devez choisir ce mode. Faites un jet de risque pour chaque figurine de votre unité (06.03).
EN SE DÉPLAÇANT :
Fuite Désespérée : Chaque figurine déplacée peut se déplacer à travers les figurines ennemies.
APRÈS S'ÊTRE DÉPLACÉE :
- Votre unité doit être non engagée.
Jusqu'à la fin du tour, sauf mention co

## 5. Vue d'ensemble des scores

Pour chaque question : le code du top-1, son score, le score du 5e, et l'**écart** entre le 1er et le 2e.

À observer :

- un **écart net** entre le 1er et le 2e suggère une réponse bien identifiée ;
- un **top-5 très resserré** signale une question ambiguë, ou des chunks qui se ressemblent tous ;
- la question **hors corpus** devrait avoir un top-1 plus bas que les autres. Mais pas forcément très bas, à cause de l'anisotropie vue au notebook 03.

In [37]:
print(f"{'top-1':>12}  {'score 1':>7}  {'score 5':>7}  {'écart 1-2':>9}  question")
for question in QUESTIONS:
    res, _, _ = rechercher(question, k=5)
    s = [r["score"] for r in res]
    print(f"{res[0]['code'] or '(sans code)':>12}  {s[0]:7.3f}  {s[-1]:7.3f}  {s[0] - s[1]:9.3f}  {question[:60]}")

       top-1  score 1  score 5  écart 1-2  question
       09.07    0.549    0.537      0.008  Que doit faire une unitée en fuite ?
       05.04    0.583    0.537      0.007  Comment alouer les dégats sur une unité composée de modèles 
       01.02    0.641    0.574      0.036  Quel est le commandement d'une unité à plusieurs figurines a
       24.27    0.617    0.532      0.045  Les pistolets peuvent-ils être utilisés au corps à corps?


## 6. Distribution des scores sur tout le corpus

On charge tous les vecteurs de la base et on calcule la similarité de la question avec **chacun** des chunks. Si la recherche est discriminante, le top-1 doit se détacher nettement de la **médiane**, c'est-à-dire du chunk « typique ». Pour une question hors corpus, on s'attend à un maximum proche du reste.

Le cas extrême, une question qui a une similarité à peu près identique avec tous les chunks, signifierait que l'encodeur ne distingue rien.

In [39]:
lignes = conn.execute(f"SELECT id, code, embedding FROM {TABLE}").fetchall()
ids = [r[0] for r in lignes]
codes = [r[1] or "(sans code)" for r in lignes]
matrice = np.stack([r[2].to_numpy() for r in lignes])  # (n_chunks, 1024), normes 1
print("matrice :", matrice.shape)

print(f"\n{'médiane':>7}  {'p90':>6}  {'p99':>6}  {'max':>6}  {'max - méd.':>10}  question")
for question in QUESTIONS:
    sims = matrice @ encodeur.encoder([question])[0]
    p50, p90, p99 = np.percentile(sims, [50, 90, 99])
    print(f"{p50:7.3f}  {p90:6.3f}  {p99:6.3f}  {sims.max():6.3f}  {sims.max() - p50:10.3f}  {question[:60]}")

matrice : (207, 1024)

médiane     p90     p99     max  max - méd.  question
  0.440   0.515   0.539   0.549       0.109  Que doit faire une unitée en fuite ?
  0.431   0.494   0.563   0.583       0.152  Comment alouer les dégats sur une unité composée de modèles 
  0.454   0.538   0.598   0.641       0.187  Quel est le commandement d'une unité à plusieurs figurines a
  0.418   0.484   0.536   0.617       0.200  Les pistolets peuvent-ils être utilisés au corps à corps?


## 7. Recherche exacte contre HNSW

- **Exacte** : numpy compare la question à tous les chunks et trie. C'est la vérité de référence.
- **HNSW** : on force l'usage de l'index (`enable_seqscan = off`) pour obtenir la recherche **approximative**.

Le **recouvrement** est la part des k résultats exacts que HNSW retrouve aussi. Sur ~200 chunks, on attend 100 %. L'écart n'apparaîtrait qu'à bien plus grande échelle, où l'on ajusterait `hnsw.ef_search`. Rappel : sans forcer, Postgres fait ici un scan séquentiel, qui est exact.

In [40]:
SQL_IDS = f"SELECT id FROM {TABLE} ORDER BY embedding <=> %s LIMIT %s"
K = 5

conn.execute("SET enable_seqscan = off")
for question in QUESTIONS:
    q = encodeur.encoder([question])[0]
    exact = [ids[i] for i in np.argsort(-(matrice @ q))[:K]]
    hnsw = [r[0] for r in conn.execute(SQL_IDS, (q, K))]
    recouvrement = len(set(exact) & set(hnsw)) / K
    print(f"recouvrement {recouvrement:4.0%}  même ordre : {exact == hnsw!s:5}  {question[:60]}")
conn.execute("RESET enable_seqscan");

recouvrement 100%  même ordre : True   Que doit faire une unitée en fuite ?
recouvrement 100%  même ordre : True   Comment alouer les dégats sur une unité composée de modèles 
recouvrement 100%  même ordre : True   Quel est le commandement d'une unité à plusieurs figurines a
recouvrement 100%  même ordre : True   Les pistolets peuvent-ils être utilisés au corps à corps?


## 8. Premier aperçu chiffré (optionnel) : hit@k

Pour chaque question, indique le ou les **codes « XX.YY » attendus**, ceux qu'un bon retrieval devrait remonter. On calcule alors :

- **hit@k** : au moins un code attendu figure-t-il dans le top-k ?
- **rang** : position du premier code attendu (1 = parfait).

Ce n'est pas encore l'évaluation (trop peu de questions, pas de gold set relu). Mais ce dictionnaire est le **brouillon de ton futur `evals/dataset.jsonl`** : autant noter tes attentes dès maintenant. Les questions sans attente (liste vide) sont ignorées.

In [41]:
ATTENDUS = {
    # "question exacte de QUESTIONS": ["XX.YY", ...],
    QUESTIONS[0]: [],
    QUESTIONS[1]: [],
    QUESTIONS[2]: [],
    QUESTIONS[3]: [],
    QUESTIONS[4]: [],
    QUESTIONS[5]: [],
}
K = 5

evaluees = {q: a for q, a in ATTENDUS.items() if a}
touches = 0
for question, attendus in evaluees.items():
    res, _, _ = rechercher(question, k=K)
    codes_top = [r["code"] for r in res]
    rangs = [i + 1 for i, c in enumerate(codes_top) if c in attendus]
    touches += bool(rangs)
    print(f"{'OK ' if rangs else 'RATÉ'}  rang {rangs[0] if rangs else '-':>2}  "
          f"attendu {attendus}  obtenu {codes_top}  | {question[:50]}")

if evaluees:
    print(f"\nhit@{K} : {touches}/{len(evaluees)} = {touches / len(evaluees):.0%}")
else:
    print("Renseigne au moins une liste de codes attendus dans ATTENDUS.")

IndexError: list index out of range

## Bilan à noter

- Les top-1 sont-ils **pertinents** ? Sur quels types de questions la recherche échoue-t-elle (synonymes, règles proches, multi-sections) ?
- Les **scores** : quelle plage pour une bonne réponse, et quelle plage pour le hors corpus ? Se recouvrent-elles ? Si oui, aucun seuil absolu ne pourra trier les résultats pertinents.
- La **latence** : encodage de la question contre requête SQL.
- Le **recouvrement** exact/HNSW.
- Les **questions** et **codes attendus** à reprendre dans le gold set.

In [ ]:
conn.close()